# Notebook 1 — QA Dataset (Task 2)

This notebook covers the **data layer for Task 2** (chatbots). The dataset is a deterministically generated set of 2,000+ English question-answer pairs across factual topics (geography, science, math, technology, history, general knowledge) plus some conversational chit-chat. Generation code lives in `task2_chatbot/data/generate_qa_dataset.py`.

> Task 1's text corpus is in a separate notebook at [`../../task1_text_generation/notebooks/01_data.ipynb`](../../task1_text_generation/notebooks/01_data.ipynb).

## Setup

In [ ]:
import os
import sys
import json
import random
from collections import Counter

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir, os.pardir))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

DATA_DIR = os.path.join(PROJECT_ROOT, "task2_chatbot", "data")
print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir:     {DATA_DIR}")

## How the dataset is built

The dataset is assembled in layers:

1. **Hand-written seed pairs** across geography, science, math, technology, history, general knowledge, and conversational chit-chat.
2. **Templated pairs** from a concept dictionary (e.g. `What is {concept}?`, `How does {concept} work?`, `What is the difference between {a} and {b}?`).
3. **Prefix variations** (e.g. "Could you tell me, what is the capital of France?") to expand surface form coverage.
4. **Rephrasings** (`What is` → `Define`/`Describe`, `How does` → `In what way does`, …).
5. **Follow-up phrasings** (`Can you elaborate on {topic}?`, `Summarize {topic} for me.`).

Duplicates are removed by normalized question text.

In [ ]:
from task2_chatbot.data.generate_qa_dataset import generate_all_qa_pairs

qa_path = os.path.join(DATA_DIR, "qa_pairs.json")

if not os.path.exists(qa_path):
    print("Generating QA pairs (seed=42)...")
    qa_pairs = generate_all_qa_pairs(seed=42)
    with open(qa_path, "w", encoding="utf-8") as f:
        json.dump(qa_pairs, f, indent=2, ensure_ascii=False)
else:
    with open(qa_path, "r", encoding="utf-8") as f:
        qa_pairs = json.load(f)

print(f"Total QA pairs: {len(qa_pairs):,}")

### Sample pairs

In [ ]:
random.seed(0)
for pair in random.sample(qa_pairs, 8):
    print(f"Q: {pair['question']}")
    print(f"A: {pair['answer']}")
    print("-" * 70)

## Dataset statistics

Question/answer length distributions set the sequence length budget for the seq2seq models.

In [ ]:
import matplotlib.pyplot as plt

q_lens = [len(p["question"].split()) for p in qa_pairs]
a_lens = [len(p["answer"].split()) for p in qa_pairs]

print(f"Question length — min={min(q_lens)}, max={max(q_lens)}, mean={sum(q_lens)/len(q_lens):.1f}")
print(f"Answer length   — min={min(a_lens)}, max={max(a_lens)}, mean={sum(a_lens)/len(a_lens):.1f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(q_lens, bins=range(1, max(q_lens) + 2), edgecolor="black", alpha=0.8)
axes[0].set_title("Question length (words)")
axes[0].set_xlabel("words"); axes[0].set_ylabel("count")

axes[1].hist(a_lens, bins=range(1, max(a_lens) + 2), edgecolor="black", alpha=0.8, color="C1")
axes[1].set_title("Answer length (words)")
axes[1].set_xlabel("words"); axes[1].set_ylabel("count")
plt.tight_layout()
plt.show()

In [ ]:
first_word_counts = Counter(p["question"].split()[0] for p in qa_pairs)
top_starters = first_word_counts.most_common(15)

plt.figure(figsize=(11, 4))
plt.bar([w for w, _ in top_starters], [c for _, c in top_starters])
plt.title("Most common question-opening words")
plt.ylabel("count")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## Summary

- 2,000+ QA pairs written to `task2_chatbot/data/qa_pairs.json` — reproducible via `seed=42`.
- Next: [`02_chatbots.ipynb`](02_chatbots.ipynb) trains the LSTM seq2seq + Transformer chatbots on these pairs.